In [ ]:
blood_vars_2 = [ 'biospec_blood_eos_abs' , 'biospec_blood_hemoglobin' ,'biospec_blood_plt_count', 'biospec_blood_wbc_count' ,\
'biospec_blood_ferritin' ,	'biospec_blood_imm_gran_abs' ,'biospec_blood_lymph_abs',  \
'biospec_blood_mono_abs',	'biospec_blood_mpv',	'biospec_blood_neut_abs',\
'biospec_blood_rdw',	'biospec_blood_hematocrit']
#for inflame analyses

In [ ]:
###CHECK SKEWNESS 
import pandas as pd
import numpy as np

skewness = (
    df[blood_vars_2]
    .apply(lambda x: pd.to_numeric(x, errors="coerce"))
    .skew()
    .sort_values(ascending=False)
)

skewness

In [ ]:
#check kurtosis
from scipy.stats import kurtosis

raw_kurt = (
    df[blood_vars_2]
    .apply(pd.to_numeric, errors="coerce")
    .apply(lambda x: kurtosis(x.dropna(), fisher=True))
    .sort_values(ascending=False)
)

raw_kurt

In [ ]:
#check outliers
def p99_outliers(series):
    p99 = series.quantile(0.99)
    return (series > p99).sum()

p99_counts = (
    df[vars_qc]
    .apply(pd.to_numeric, errors='coerce')
    .apply(p99_outliers)
    .rename('p99_outlier_count')
)

p99_counts


In [ ]:
#plan to remove RDW outliers > 99%ile, after visual and data assessments
rdw = pd.to_numeric(df['biospec_blood_rdw'], errors='coerce')

rdw_99 = rdw.quantile(0.99)
rdw_995 = rdw.quantile(0.995)

rdw_99, rdw_995


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(rdw.dropna(), bins=50)
plt.axvline(rdw_99, linestyle='--')
plt.title('RDW distribution (raw)')
plt.xlabel('RDW')
plt.ylabel('Count')
plt.show()


In [ ]:
rdw_trim = rdw[rdw <= rdw_99]

plt.figure()
plt.hist(rdw_trim.dropna(), bins=50)
plt.title('RDW distribution (after removing >99th percentile)')
plt.xlabel('RDW')
plt.ylabel('Count')
plt.show()


In [ ]:

# ----------------------------
# 0) Define variables
# ----------------------------
v_neut     = 'biospec_blood_neut_abs'
v_lymph    = 'biospec_blood_lymph_abs'
v_rdw      = 'biospec_blood_rdw'
v_wbc      = 'biospec_blood_wbc_count'

# Ensure numeric
for v in [v_ferritin, v_neut, v_lymph, v_rdw, v_wbc]:
    df[v] = pd.to_numeric(df[v], errors='coerce')

# ----------------------------
# 1) RDW: trim >99th percentile (keep raw scale)
# ----------------------------
rdw_cut99 = df[v_rdw].quantile(0.99)
df['rdw_trim'] = df[v_rdw].where(df[v_rdw] <= rdw_cut99)

print(f"RDW 99th percentile cutoff: {rdw_cut99:.3f}")
print("RDW removed (set to NaN):", int((df[v_rdw] > rdw_cut99).sum()))


In [ ]:
# ----------------------------
# 3) Lymphocytes: keep raw (but ensure >0 for ratio)
# ----------------------------
df['lymph_raw'] = df[v_lymph]

# ----------------------------
# 4) N:L ratio (use raw neut & raw lymph)
# ----------------------------
# Create ratio only where both components are present and lymph > 0
valid_ratio = df[v_neut].notna() & df['lymph_raw'].notna() & (df['lymph_raw'] > 0)

df['nl_ratio'] = np.nan
df.loc[valid_ratio, 'nl_ratio'] = df.loc[valid_ratio, v_neut] / df.loc[valid_ratio, 'lymph_raw']

In [ ]:
nl = df['nl_ratio']

nl.describe()


In [ ]:
#nl kurtosis and skew
nl_nonan = nl.dropna()

{
    "n": len(nl_nonan),
    "skew": float(skew(nl_nonan)),
    "kurtosis": float(kurtosis(nl_nonan, fisher=True))
}


In [ ]:
#graph NL

plt.figure()
plt.hist(nl_nonan, bins=50)
plt.title("N:L ratio (raw)")
plt.xlabel("neut / lymph")
plt.ylabel("Count")
plt.show()


{
    "skew": float(skew(nl_nonan)),
    "kurtosis": float(kurtosis(nl_nonan, fisher=True))
}

In [ ]:

# ----------------------------
# N:L ratio (final version)
# ----------------------------

# Ensure numeric
df[v_neut]  = pd.to_numeric(df[v_neut], errors='coerce')
df[v_lymph] = pd.to_numeric(df[v_lymph], errors='coerce')

# Compute N:L ratio (only where valid)
df['nl_ratio'] = np.nan
valid = df[v_neut].notna() & df[v_lymph].notna() & (df[v_lymph] > 0)
df.loc[valid, 'nl_ratio'] = df.loc[valid, v_neut] / df.loc[valid, v_lymph]

# ----------------------------
# Diagnostics summary (FIXED)
# ----------------------------

diag_table = pd.DataFrame({
    'variable': [
        'wbc_raw',
        'rdw_trim',
        'nl_ratio',
        'plt'
    ],
    'n': [
        df[v_wbc].notna().sum(),
        df['rdw_trim'].notna().sum(),
        df['ferritin'].notna().sum(),
        df['log_ferritin'].notna().sum(),
        df['log_neut'].notna().sum(),
        df['nl_ratio'].notna().sum(),
        df['log_nl_ratio'].notna().sum(),
        df['plt'].notna().sum()
    ],
    'skew': [
        skew(df[v_wbc].dropna()),
        skew(df['rdw_trim'].dropna()),
        skew(df['ferritin'].dropna()),
        skew(df['log_ferritin'].dropna()),
        skew(df['log_neut'].dropna()),
        skew(df['nl_ratio'].dropna()),
        skew(df['log_nl_ratio'].dropna()),
        skew(df['plt'].dropna())
    ],
    'kurtosis': [
        kurtosis(df[v_wbc].dropna(), fisher=True),
        kurtosis(df['rdw_trim'].dropna(), fisher=True),
        kurtosis(df['ferritin'].dropna(), fisher=True),
        kurtosis(df['log_ferritin'].dropna(), fisher=True),
        kurtosis(df['log_neut'].dropna(), fisher=True),
        kurtosis(df['nl_ratio'].dropna(), fisher=True),
        kurtosis(df['log_nl_ratio'].dropna(), fisher=True),
        kurtosis(df['plt'].dropna(), fisher=True)
    ]
})

diag_table


In [ ]:
#assessing residuals
resid = model.residuals
fitted = model.fits
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(fitted, resid)
plt.axhline(0)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title(f"Residuals vs Fitted: {lab}")
plt.show()


In [2283]:

blood_vars_2 = ['wbc_raw', 'rdw_trim', 'nl_ratio', 'plt', 'biospec_blood_hemoglobin']
#hemoglobin just kept as sensitivity analysis

In [ ]:
mediators = ['wbc_raw', 'rdw_trim','nl_ratio','plt']

df_mediation = df.dropna(subset=mediators)

df_mediation.shape


In [2290]:
base_columns = ['src_subject_id', 'eventname', 'BMI', 'ACE_total', 'sex', 'interview_age', 'site_id_l',
                'Puberty_PDS', 'highest_edu', 'ACE_sum', 'race' ,
                'demo_ethn_v2', 'ACE_PA_sum_count', 'ACE_SA_sum_count', 'ACE_N_sum_count', 'ACE_EA_sum_count', 'ACE_D_sum_count',
                'ACE_MH_sum_count', 'ACE_DV_sum_count', 'ACE_F_sum_count', 'ACE_NV_sum_count',
                'ACE_J2_sum_count', 'ACE_B_sum_count', 'ACE_SS_sum_count',
                'ACE_PA_count', 'ACE_SA_count', 'ACE_N_count', 'ACE_EA_count', 'ACE_D_count',
                'ACE_MH_count', 'ACE_DV_count', 'ACE_F_count', 'ACE_NV_count',
                'ACE_J2_count', 'ACE_B_count', 'ACE_SS_count', 'weight_status'
                ]
#demographic information, ACEs


In [ ]:
# Keep only participants with complete blood data
data_df_blood = data_df_blood.loc[data_df_blood.index.isin(labs_complete.index)]

print("Kids with complete blood data (kept):", data_df_blood.shape[0])


In [1917]:
columns_to_standardize = ['ACE_total', 'interview_age', 'Puberty_PDS','ACE_sum','wbc_raw', 'rdw_trim', 'nl_ratio', 'plt']    
                  

# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the specified columns, replacing them with their standardized values
data_df_blood[columns_to_standardize] = scaler.fit_transform(data_df_blood[columns_to_standardize])

In [ ]:
##ACE-BMI
formula = 'BMI ~ ACE_total + sex + interview_age + Puberty_PDS + highest_edu + (1|site_id_l)'

model = Lmer(formula, data=data_df_blood)  # 
model_fit = model.fit(factors={
    'sex': {"1.0": -1, "2.0": 1},
    'highest_edu': {"5.0": -1, "4.0": 0.25, "3.0": 0.25, "2.0": 0.25, "1.0": 0.25}
})

display(model_fit)


In [ ]:
##impact ace on inflammation
results = []
for lab in blood_vars_2:
    formula = f'{lab} ~ ACE_total + sex + interview_age + Puberty_PDS + highest_edu + (1|site_id_l)'
    print(f"\nRunning model with {lab}...")
    
    model = Lmer(formula, data=data_df_blood)
    fit = model.fit(factors={
        'sex': {"1.0": -1, "2.0": 1},
        'highest_edu': {"5.0": -1, "4.0": 0.25, "3.0": 0.25, "2.0": 0.25, "1.0": 0.25}
    })
    display(fit)  
##ONLY RDW

In [ ]:
##impact inflammation on BMI 
results = []
for lab in blood_vars_2:
    formula = f'BMI ~ {lab} + sex + interview_age + Puberty_PDS + highest_edu + (1|site_id_l)'
    print(f"\nRunning model with {lab}...")
    
    model = Lmer(formula, data=data_df_blood)
    fit = model.fit(factors={
        'sex': {"1.0": -1, "2.0": 1},
        'highest_edu': {"5.0": -1, "4.0": 0.25, "3.0": 0.25, "2.0": 0.25, "1.0": 0.25}
    })
    display(fit)  


In [ ]:
#additional sensitivity analysis: hgb, race

# RDW only significant - mediation
#additional sensitivity analysis: site

In [ ]:
# --- Mediation---
blood_vars = ['rdw_trim']
covars = ['sex', 'interview_age', 'Puberty_PDS', 'highest_edu']
analysis_cols = ['ACE_total', 'BMI'] + blood_vars + covars + ['site_id_l']

df = data_df_blood.loc[:, analysis_cols].dropna().copy()

site_dummies = pd.get_dummies(df['site_id_l'].astype('category'),
                              prefix='site', drop_first=True, dtype=float)

df = pd.concat([df.drop(columns=['site_id_l']), site_dummies], axis=1)

# Ensure all model columns are numeric floats
model_cols = ['ACE_total', 'BMI'] + blood_vars + covars + list(site_dummies.columns)
df[model_cols] = df[model_cols].apply(pd.to_numeric, errors='coerce')

covars_plus_site = covars + list(site_dummies.columns)

p = Process(
    data=df,
    model=4,
    x='ACE_total',
    y='BMI',
    m=['rdw_trim'],
    controls=covars_plus_site,
    controls_in='all',
    n_boot=10000,
    seed=2025,
    hc3=True,
    suppr_init=True,
)

p.summary()
